# Huang Lab — Tissue-on-a-Chip MAE + Detection Training

**Pipeline:**
1. Multi-encoder MAE pretraining (focused images + hybrid projections + z-stack volumes)
2. Deformable-DETR detection fine-tuning on bounding box labels

**Before running:**
- Runtime → Change runtime type → **T4 GPU**
- Make sure `250918_Deepmind_CV_Collaboration` is added as a shortcut in your My Drive
- Have your W&B API key ready from https://wandb.ai/authorize

## 1. Check GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_ROOT = '/content/drive/MyDrive/250918_Deepmind_CV_Collaboration'
assert os.path.isdir(DATA_ROOT), (
    f'Data folder not found at {DATA_ROOT}.\n'
    'Please right-click 250918_Deepmind_CV_Collaboration in Google Drive '
    '(Shared with me) and choose Organize -> Add shortcut -> My Drive'
)
print('Data root found:', DATA_ROOT)
print(os.listdir(DATA_ROOT))

## 3. Clone repo and install dependencies

In [ ]:
# Clone the repo (from main after PR is merged)
!git clone https://github.com/racyun/Huang-Lab-Work.git /content/Huang-Lab-Work
%cd /content/Huang-Lab-Work

In [ ]:
# Install all dependencies
!pip install -q -r requirements.txt
!pip install -q transformers accelerate timm
print('Dependencies installed.')

## 4. W&B login

In [ ]:
import wandb
wandb.login()  # Paste your API key from https://wandb.ai/authorize

## 5. Write Colab config (full 222-well dataset, GPU settings)

In [ ]:
colab_config = f"""
# Auto-generated Colab config — do not commit.

dataset:
  well_prefix: "W"
  well_count: 222
  zstack_subdir: "P00001"
  expected_z_slices: 140
  hybrid_folder_template: "hybrid_results_{{well_id}}"
  focus_filename_glob: "*focus_stacked.tif"
  cache_dir: "/content/huang_lab_cache"
  resize:
    zstack:  [64, 64]   # cache at 64x64 to fit in Colab RAM; model upsamples
    focused: [224, 224]
    hybrid:  [224, 224]

  splits:
    - name: "900kpa"
      stiffness_kpa: 900.0
      zstack_root:  "{DATA_ROOT}/250811_Athchip_noninflam_900kPa"
      focused_root: "{DATA_ROOT}/20251027_2123__FocusStack_250811_Athchip_noninflam_900kPa"
      hybrid_root:  "{DATA_ROOT}/250811_Athchip_noninflam_900kPa_HybridResults"
      labels_root:  "{DATA_ROOT}/bbox_txt_for_training/900kPa"

    - name: "5kpa"
      stiffness_kpa: 5.0
      zstack_root:  "{DATA_ROOT}/250814_Athchip_non-inflam_5kPa"
      focused_root: "{DATA_ROOT}/20251027_2215__FocusStack_250814_Athchip_non-inflam_5kPa"
      hybrid_root:  "{DATA_ROOT}/250814_Athchip_non-inflam_5kPa_HybridResults"
      labels_root:  "{DATA_ROOT}/bbox_txt_for_training/5kPa"

training:
  output_dir: "/content/outputs"
  batch_size: 4          # T4 has 16GB VRAM; increase to 8 if memory allows
  num_workers: 2
  epochs: 50
  lr: 1.5e-4
  warmup_epochs: 5
  amp: true              # mixed precision on GPU
  grad_clip: 1.0

detection:
  epochs: 50
  batch_size: 4
  num_workers: 2
  amp: true
  train_image_short_side: 800   # full resolution now that we have GPU
  conf_threshold: 0.3

wandb:
  enabled: true
  project: "huang-lab-tissue-chip"
  entity: null           # uses your logged-in account
  log_freq: 5
"""

with open('config/colab.yaml', 'w') as f:
    f.write(colab_config)
print('config/colab.yaml written.')
print(colab_config)

## 6. Verify data loads correctly (one batch)

In [ ]:
!python3 scripts/train_pretrain.py \
    --config config/default.yaml \
    --local-config config/colab.yaml \
    --inspect-data

## 7. Pretrain — Multi-encoder MAE

Trains three encoders jointly:
- **Focused encoder** — 2D ViT on focus-stacked images
- **Hybrid encoder** — 2D ViT on hybrid projections  
- **Volume encoder** — VideoMAE-style 3D ViT on z-stacks

Stiffness (kPa) is injected as a conditioning signal into every patch token.

Logs to W&B: `train/loss`, `train/loss_focused`, `train/loss_hybrid`, `train/loss_volume`, per-encoder LRs.

In [ ]:
!python3 scripts/train_pretrain.py \
    --config config/default.yaml \
    --local-config config/colab.yaml \
    --train \
    --wandb \
    --wandb-run-name "full-222well-pretrain-50ep"

## 8. Detection fine-tuning — Deformable-DETR

Fine-tunes Deformable-DETR on the bounding box labels.

Logs to W&B: `detect/loss`, `eval/mAP`, `eval/AP50`, `eval/AP75`, `eval/mean_iou`.

**To use the pretrained MAE encoder as backbone** (optional): set `mae_encoder_ckpt` in the config to point to the checkpoint saved in `/content/outputs/pretrain/`.

In [ ]:
# Find the latest pretrain checkpoint
import glob
ckpts = sorted(glob.glob('/content/outputs/pretrain/*.pth'))
if ckpts:
    print('Latest pretrain checkpoint:', ckpts[-1])
else:
    print('No checkpoint found — running detection from COCO weights only')

In [ ]:
!python3 scripts/train_detect.py \
    --config config/default.yaml \
    --local-config config/colab.yaml \
    --wandb \
    --wandb-run-name "full-222well-detect-50ep"

## 9. Save outputs to Drive (so they persist after Colab disconnects)

In [ ]:
import shutil, os

SAVE_DIR = f'{DATA_ROOT}/colab_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copytree('/content/outputs', SAVE_DIR, dirs_exist_ok=True)
print(f'Outputs saved to Google Drive: {SAVE_DIR}')
print(os.listdir(SAVE_DIR))

## Tips

**If Colab disconnects mid-run:**
- Checkpoints are saved every epoch to `/content/outputs/`
- Resume pretraining: add `--resume /content/outputs/pretrain/checkpoint_epochN.pth` to the pretrain command
- The cache in `/content/huang_lab_cache/` will be rebuilt on reconnect (first epoch slow, rest fast)

**Expected training times on T4 GPU:**
- Pretrain epoch (222 wells, batch 4): ~8–15 min/epoch
- Detection epoch (222 wells, batch 4, 800px): ~5–10 min/epoch
- 50 epochs total: ~10–12 hours (use Colab Pro for longer sessions)

**Expected metrics after 50 epochs:**
- MAE loss: should drop to < 0.01
- Detection AP50: realistically 0.3–0.6 (cells are compact, uniform objects)
- Detection mAP@[0.5:0.95]: typically 0.15–0.35

**W&B dashboard:** https://wandb.ai/models-fusionai/huang-lab-tissue-chip